# ONNX In-Database Embeddings with Oracle AI Database 26ai

This notebook demonstrates the **Oracle-first ONNX workflow** in which an embedding model is loaded into Oracle AI Database and all embedding generation happens **inside the database** using `VECTOR_EMBEDDING()`.

No external embedding API (OpenAI, Cohere, etc.) is needed — the ONNX model runs entirely within Oracle.

The notebook covers:
- loading the augmented all-MiniLM-L12-v2 ONNX model into Oracle
- generating embeddings in-database with `VECTOR_EMBEDDING()`
- storing and querying vectors with Oracle AI Vector Search
- LangChain integration using Oracle-native components

## What You Will Learn

By the end of this notebook you will know how to:

1. Connect to Oracle AI Database 26ai from Python
2. Load an ONNX embedding model into Oracle with `DBMS_VECTOR.LOAD_ONNX_MODEL`
3. Generate embeddings **inside the database** with `VECTOR_EMBEDDING()`
4. Create tables with `VECTOR` columns and insert documents with in-database embeddings
5. Run semantic similarity search entirely in Oracle SQL
6. Integrate the in-database ONNX workflow with LangChain using `OracleEmbeddings` and `OracleVS`

## Prerequisites

Before running this notebook, make sure you have:

- **Python 3.10+** with `oracledb`, `langchain-oracledb`, and `langchain-community` (installed automatically in the next cell)
- **Oracle AI Database 26ai** running in Docker:
  ```bash
  docker run -d --name oracle-26ai \
    -p 1521:1521 \
    -e ORACLE_PWD=YourSysPassword \
    -v /path/to/onnx_models:/opt/oracle/onnx_models \
    container-registry.oracle.com/database/free:latest
  ```
- **The augmented all-MiniLM-L12-v2 ONNX model** — downloaded automatically by Step 1b if not already present
- A database user with `DB_DEVELOPER_ROLE`, `CREATE MINING MODEL`, and `READ`/`WRITE` on the ONNX directory — created automatically by the Admin Setup cell

> **Important:** Oracle requires an *augmented* ONNX model that bundles tokenization and post-processing into the ONNX graph. Step 1b downloads the correct pre-built file directly from Oracle's public OCI bucket.

In [1]:
import subprocess
import sys

result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "oracledb", "python-dotenv", "pandas", "numpy",
     "langchain", "langchain-core", "langchain-community", "langchain-oracledb"],
    capture_output=True, text=True
)
print("Packages installed." if result.returncode == 0 else f"Install failed: {result.stderr}")

Packages installed.


## Configuration

Create a `.env` file in the same directory as this notebook:

```
ORACLE_USER=testuser
ORACLE_PASSWORD=TestPass123
ORACLE_DSN=localhost:1521/FREEPDB1

# ONNX model settings
ORACLE_MODEL_NAME=ALL_MINILM_L12_V2
ORACLE_DIRECTORY_NAME=ONNX_DIR
ORACLE_ONNX_FILE=all_MiniLM_L12_v2.onnx
```

> **Docker setup:** This notebook expects Oracle AI Database 26ai running in Docker with the ONNX model directory mounted at `/opt/oracle/onnx_models`. See the Admin Setup Reference section below for the full `docker run` command.

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

ORACLE_USER = os.getenv("ORACLE_USER")
ORACLE_PASSWORD = os.getenv("ORACLE_PASSWORD")
ORACLE_DSN = os.getenv("ORACLE_DSN")

MODEL_NAME = os.getenv("ORACLE_MODEL_NAME", "ALL_MINILM_L12_V2")
DIRECTORY_NAME = os.getenv("ORACLE_DIRECTORY_NAME", "ONNX_DIR")
ONNX_FILE = os.getenv("ORACLE_ONNX_FILE", "all_MiniLM_L12_v2.onnx")

print(f"User:      {ORACLE_USER}")
print(f"DSN:       {ORACLE_DSN}")
print(f"Model:     {MODEL_NAME}")
print(f"Directory: {DIRECTORY_NAME}")
print(f"ONNX file: {ONNX_FILE}")

User:      testuser
DSN:       localhost:1522/FREEPDB1
Model:     ALL_MINILM_L12_V2
Directory: ONNX_DIR
ONNX file: all_MiniLM_L12_v2.onnx


## Admin Setup — One-Time Setup

Run the cell below **once** to create the database user, grant privileges, and register the ONNX model directory.

**Start the Oracle 26ai Free container** (if not already running):

```bash
docker run -d --name oracle-26ai \
  -p 1521:1521 \
  -e ORACLE_PWD=YourSysPassword \
  -v /path/to/onnx_models:/opt/oracle/onnx_models \
  container-registry.oracle.com/database/free:latest
```

> Uses `docker` by default; falls back to `podman` if Docker is not found.


In [3]:
# Automated admin setup — run once to create user, grants, and directory.
# Requires the oracle-26ai container to be running and healthy.

import subprocess, shutil

CONTAINER_CLI = "docker" if shutil.which("docker") else "podman"
print(f"Using container runtime: {CONTAINER_CLI}")

admin_sql = """
ALTER SESSION SET CONTAINER = FREEPDB1;

-- Create user (or unlock if exists)
DECLARE
    v_count NUMBER;
BEGIN
    SELECT COUNT(*) INTO v_count FROM dba_users WHERE username = 'TESTUSER';
    IF v_count = 0 THEN
        EXECUTE IMMEDIATE 'CREATE USER testuser IDENTIFIED BY "TestPass123" DEFAULT TABLESPACE users TEMPORARY TABLESPACE temp QUOTA UNLIMITED ON users';
        DBMS_OUTPUT.PUT_LINE('User TESTUSER created.');
    ELSE
        EXECUTE IMMEDIATE 'ALTER USER testuser IDENTIFIED BY "TestPass123" ACCOUNT UNLOCK';
        DBMS_OUTPUT.PUT_LINE('User TESTUSER already exists — password reset and account unlocked.');
    END IF;
END;
/

GRANT DB_DEVELOPER_ROLE TO testuser;
GRANT CREATE MINING MODEL TO testuser;
GRANT CONNECT, RESOURCE TO testuser;

CREATE OR REPLACE DIRECTORY ONNX_DIR AS '/opt/oracle/onnx_models';
GRANT READ, WRITE ON DIRECTORY ONNX_DIR TO testuser;

-- Verify
SELECT 'User: ' || username AS status FROM dba_users WHERE username = 'TESTUSER';
SELECT 'Directory: ' || directory_name || ' -> ' || directory_path AS status FROM dba_directories WHERE directory_name = 'ONNX_DIR';

EXIT;
"""

result = subprocess.run(
    [CONTAINER_CLI, "exec", "-i", "oracle-26ai", "bash", "-c", "sqlplus -s / as sysdba"],
    input=admin_sql, capture_output=True, text=True
)

if result.returncode == 0:
    for line in result.stdout.strip().splitlines():
        line = line.strip()
        if line and not line.startswith("Session") and not line.startswith("Grant") and "row" not in line.lower():
            print(line)
    print("\nAdmin setup complete.")
else:
    print(f"Error: {result.stderr}")


Using container runtime: podman
PL/SQL procedure successfully completed.
Directory created.
STATUS
--------------------------------------------------------------------------------
User: TESTUSER
STATUS
--------------------------------------------------------------------------------
Directory: ONNX_DIR -> /opt/oracle/onnx_models

Admin setup complete.


## Step 1 — Connect to Oracle AI Database

We use `python-oracledb` in Thin mode (no Oracle Client libraries needed).

In [4]:
import oracledb

conn = oracledb.connect(user=ORACLE_USER, password=ORACLE_PASSWORD, dsn=ORACLE_DSN)
print(f"Connected to Oracle AI Database {conn.version}")

Connected to Oracle AI Database 23.26.1.0.0


## Helper Function

A small utility for running SQL and returning results as a DataFrame.

In [5]:
import pandas as pd

def run_sql(sql, params=None, fetch=False, many=False, data=None):
    """Execute SQL against Oracle Database."""
    with conn.cursor() as cur:
        if many and data:
            cur.executemany(sql, data)
        elif params:
            cur.execute(sql, params)
        else:
            cur.execute(sql)

        if fetch:
            cols = [c[0] for c in cur.description]
            return pd.DataFrame(cur.fetchall(), columns=cols)

    conn.commit()

## Step 1b — Download the Augmented ONNX Model (if needed)

Oracle requires an *augmented* ONNX model that bundles tokenization into the graph — a raw HuggingFace export will not work.

The cell below downloads the official pre-built model from Oracle's public OCI bucket and copies it into the Docker container's model directory. **Skip this if the file is already present.**

In [6]:
import subprocess, urllib.request, urllib.error, tempfile, os, zipfile, pathlib, shutil

ORACLE_MODEL_URL = (
    "https://adwc4pm.objectstorage.us-ashburn-1.oci.customer-oci.com"
    "/p/TtH6hL2y25EypZ0-rrczRZ1aXp7v1ONbRBfCiT-BDBN8WLKQ3lgyW6RxCfIFLdA6"
    "/n/adwc4pm/b/OML-ai-models/o/all_MiniLM_L12_v2_augmented.zip"
)
ORACLE_MODEL_DOCS_URL = (
    "https://docs.oracle.com/en/database/oracle/oracle-database/23/vecse/"
    "import-onnx-models-oracle-database-end-end-example.html"
)
CONTAINER_MODEL_DIR = "/opt/oracle/onnx_models"
CONTAINER_NAME = "oracle-26ai"

CONTAINER_CLI = "docker" if shutil.which("docker") else "podman"
print(f"Using container runtime: {CONTAINER_CLI}")

def _file_exists_in_container(filename):
    result = subprocess.run(
        [CONTAINER_CLI, "exec", CONTAINER_NAME, "test", "-f", f"{CONTAINER_MODEL_DIR}/{filename}"],
        capture_output=True
    )
    return result.returncode == 0

if _file_exists_in_container(ONNX_FILE):
    print(f"\'{ONNX_FILE}\' already present in container — skipping download.")
else:
    print("Downloading Oracle augmented all-MiniLM-L12-v2 ONNX model (~117 MB)...")
    with tempfile.TemporaryDirectory() as tmpdir:
        zip_path = os.path.join(tmpdir, "model.zip")

        try:
            def _progress(block_count, block_size, total_size):
                if total_size > 0:
                    pct = min(100, block_count * block_size * 100 // total_size)
                    if pct % 20 == 0:
                        print(f"  {pct}%...", end=" ", flush=True)

            urllib.request.urlretrieve(ORACLE_MODEL_URL, zip_path, _progress)
            print("done.")

        except urllib.error.URLError as e:
            print(
                f"\nDownload failed: {e}\n\n"
                "The pre-signed download URL may have expired.\n"
                "Please download the model manually from the Oracle docs:\n"
                f"  {ORACLE_MODEL_DOCS_URL}\n\n"
                "Then copy the .onnx file into the container with:\n"
                f"  {CONTAINER_CLI} cp all_MiniLM_L12_v2.onnx "
                f"{CONTAINER_NAME}:{CONTAINER_MODEL_DIR}/\n"
                "and re-run this cell."
            )
            raise

        print("Extracting...")
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(tmpdir)

        onnx_path = os.path.join(tmpdir, ONNX_FILE)
        if not os.path.exists(onnx_path):
            candidates = list(pathlib.Path(tmpdir).glob("*.onnx"))
            if not candidates:
                raise FileNotFoundError("No .onnx file found in downloaded zip.")
            onnx_path = str(candidates[0])

        print(f"Copying \'{os.path.basename(onnx_path)}\' into container \'{CONTAINER_NAME}\'...")
        cp = subprocess.run(
            [CONTAINER_CLI, "cp", onnx_path, f"{CONTAINER_NAME}:{CONTAINER_MODEL_DIR}/{ONNX_FILE}"],
            capture_output=True, text=True
        )
        if cp.returncode != 0:
            raise RuntimeError(f"{CONTAINER_CLI} cp failed: {cp.stderr}")

        chmod_result = subprocess.run(
            [CONTAINER_CLI, "exec", CONTAINER_NAME, "chmod", "644", f"{CONTAINER_MODEL_DIR}/{ONNX_FILE}"],
            capture_output=True, text=True
        )
        if chmod_result.returncode != 0:
            print("chmod could not be applied inside the container — file may still be readable by Oracle.")

    print(f"\nModel \'{ONNX_FILE}\' is ready in the container.")


Using container runtime: podman
'all_MiniLM_L12_v2.onnx' already present in container — skipping download.


## Step 2 — Load the ONNX Model into Oracle

Oracle AI Database supports importing ONNX models with `DBMS_VECTOR.LOAD_ONNX_MODEL()`.

The pre-built augmented all-MiniLM-L12-v2 model includes tokenization and post-processing directly in the ONNX graph, so Oracle can accept raw text and return embedding vectors.

This cell checks whether the model is already loaded, and loads it if not.

In [7]:
model_check = run_sql(
    "SELECT COUNT(*) AS cnt FROM USER_MINING_MODELS WHERE MODEL_NAME = UPPER(:model_name)",
    params={"model_name": MODEL_NAME},
    fetch=True
)
model_exists = model_check["CNT"].iloc[0] > 0

if model_exists:
    print(f"Model '{MODEL_NAME}' is already loaded in Oracle.")
else:
    print(f"Loading ONNX model '{ONNX_FILE}' from directory '{DIRECTORY_NAME}'...")
    load_sql = """
    BEGIN
        DBMS_VECTOR.LOAD_ONNX_MODEL(
            directory  => :dir_name,
            file_name  => :onnx_file,
            model_name => :model_name,
            metadata   => JSON('{"function":"embedding","embeddingOutput":"embedding","input":{"input":["DATA"]}}')
        );
    END;
    """
    with conn.cursor() as cur:
        cur.execute(load_sql, {
            "dir_name": DIRECTORY_NAME,
            "onnx_file": ONNX_FILE,
            "model_name": MODEL_NAME
        })
    print(f"Model '{MODEL_NAME}' loaded successfully.")

Model 'ALL_MINILM_L12_V2' is already loaded in Oracle.


## Step 3 — Verify the Loaded Model

Confirm that Oracle recognizes the model as an ONNX embedding model.

In [8]:
df_model = run_sql(
    "SELECT model_name, algorithm, mining_function FROM user_mining_models WHERE model_name = :m",
    params={"m": MODEL_NAME},
    fetch=True
)
df_model

,MODEL_NAME,ALGORITHM,MINING_FUNCTION
0,ALL_MINILM_L12_V2,ONNX,EMBEDDING


## Step 4 — Generate an Embedding with VECTOR_EMBEDDING

`VECTOR_EMBEDDING()` is the SQL function that runs the ONNX model inside Oracle. It takes raw text as input and returns a vector.

In [9]:
sample_text = "Oracle AI Database can generate embeddings inside the database using an ONNX model."

with conn.cursor() as cur:
    cur.execute(
        f"SELECT VECTOR_EMBEDDING({MODEL_NAME} USING :text AS DATA) AS embedding FROM dual",
        {"text": sample_text}
    )
    embedding = cur.fetchone()[0]

print(f"Input:     {sample_text}")
print(f"Type:      {type(embedding).__name__}")
print(f"Dimension: {len(embedding)}")
print(f"First 5:   {list(embedding[:5])}")

Input:     Oracle AI Database can generate embeddings inside the database using an ONNX model.
Type:      array
Dimension: 384
First 5:   [0.007905904203653336, -0.11100003123283386, -0.02965095452964306, 0.00038876335020177066, -0.0694665014743805]


## Step 5 — Create a Table for Semantic Search

We create a table with a `VECTOR(384, FLOAT32)` column to store the embeddings generated by the 384-dimensional MiniLM model.

In [10]:
run_sql("""
BEGIN
    EXECUTE IMMEDIATE 'DROP TABLE onnx_docs PURGE';
EXCEPTION
    WHEN OTHERS THEN IF SQLCODE != -942 THEN RAISE; END IF;
END;
""")

run_sql("""
CREATE TABLE onnx_docs (
    id        NUMBER GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
    category  VARCHAR2(100),
    doc_text  CLOB,
    embedding VECTOR(384, FLOAT32)
)
""")

print("Table 'onnx_docs' created with VECTOR(384, FLOAT32) column.")

Table 'onnx_docs' created with VECTOR(384, FLOAT32) column.


## Step 6 — Insert Documents with In-Database Embeddings

Each document is inserted with its embedding generated **inside Oracle** using `VECTOR_EMBEDDING()`.

No Python-side embedding call is needed — the database handles everything.

In [11]:
documents = [
    {"category": "database",
     "doc_text": "Oracle AI Database supports AI Vector Search for semantic retrieval."},
    {"category": "database",
     "doc_text": "Embedding models in ONNX format can be loaded directly into Oracle Database."},
    {"category": "framework",
     "doc_text": "LangChain can integrate with Oracle AI Vector Search for application development."},
    {"category": "search",
     "doc_text": "Semantic search compares vector representations instead of matching keywords."},
    {"category": "architecture",
     "doc_text": "In-database embedding reduces architectural complexity by keeping inference close to data."},
]

insert_sql = f"""
INSERT INTO onnx_docs (category, doc_text, embedding)
VALUES (:category, :doc_text, VECTOR_EMBEDDING({MODEL_NAME} USING :doc_text AS DATA))
"""

with conn.cursor() as cur:
    for doc in documents:
        cur.execute(insert_sql, doc)
conn.commit()

print(f"Inserted {len(documents)} documents with in-database ONNX embeddings.")

Inserted 5 documents with in-database ONNX embeddings.


## Step 7 — Inspect the Stored Documents

In [12]:
df_docs = run_sql(
    """
    SELECT id, category, DBMS_LOB.SUBSTR(doc_text, 120, 1) AS preview
    FROM onnx_docs
    ORDER BY id
    """,
    fetch=True
)
df_docs

,ID,CATEGORY,PREVIEW
0,1,database,Oracle AI Database supports AI Vector Search f...
1,2,database,Embedding models in ONNX format can be loaded ...
2,3,framework,LangChain can integrate with Oracle AI Vector ...
3,4,search,Semantic search compares vector representation...
4,5,architecture,In-database embedding reduces architectural co...


## Step 8 — Semantic Similarity Search in Oracle

Both the query embedding and the distance computation happen **inside Oracle SQL**.

The query text is embedded on-the-fly with `VECTOR_EMBEDDING()` and compared against stored vectors using `VECTOR_DISTANCE()` with cosine similarity.

In [13]:
query_text = "How do I use Oracle Database for semantic search with embeddings?"

similarity_sql = f"""
SELECT
    id,
    category,
    DBMS_LOB.SUBSTR(doc_text, 200, 1) AS doc_preview,
    VECTOR_DISTANCE(
        embedding,
        VECTOR_EMBEDDING({MODEL_NAME} USING :query AS DATA),
        COSINE
    ) AS distance
FROM onnx_docs
ORDER BY distance
FETCH FIRST 3 ROWS ONLY
"""

df_results = run_sql(similarity_sql, params={"query": query_text}, fetch=True)

print(f"Query: {query_text}\n")
df_results

Query: How do I use Oracle Database for semantic search with embeddings?



,ID,CATEGORY,DOC_PREVIEW,DISTANCE
0,1,database,Oracle AI Database supports AI Vector Search f...,0.311266
1,2,database,Embedding models in ONNX format can be loaded ...,0.395660
2,3,framework,LangChain can integrate with Oracle AI Vector ...,0.464288


## Interpreting the Results

The smaller the cosine distance, the more semantically similar the document is to the query.

This workflow does **not** rely on keyword matching — Oracle compares vector representations generated by the same ONNX model, entirely within the database.

## Step 9 — Test Multiple Semantic Queries

We run several queries to verify that semantic ranking is consistent across different intents.

In [14]:
test_queries = [
    "Which Oracle feature helps semantic retrieval?",
    "Can I store embeddings in the database?",
    "How does LangChain work with Oracle vectors?",
    "Why are ONNX models useful here?"
]

for q in test_queries:
    print("=" * 100)
    print(f"QUERY: {q}")
    print("-" * 100)

    df = run_sql(
        f"""
        SELECT
            DBMS_LOB.SUBSTR(doc_text, 100, 1) AS doc_preview,
            VECTOR_DISTANCE(
                embedding,
                VECTOR_EMBEDDING({MODEL_NAME} USING :query AS DATA),
                COSINE
            ) AS distance
        FROM onnx_docs
        ORDER BY distance
        FETCH FIRST 3 ROWS ONLY
        """,
        params={"query": q},
        fetch=True
    )
    for _, row in df.iterrows():
        print(f"  [{row['DISTANCE']:.4f}] {row['DOC_PREVIEW']}")
    print()

QUERY: Which Oracle feature helps semantic retrieval?
----------------------------------------------------------------------------------------------------
  [0.2662] Oracle AI Database supports AI Vector Search for semantic retrieval.
  [0.4589] LangChain can integrate with Oracle AI Vector Search for application development.
  [0.4748] Semantic search compares vector representations instead of matching keywords.

QUERY: Can I store embeddings in the database?
----------------------------------------------------------------------------------------------------
  [0.4074] In-database embedding reduces architectural complexity by keeping inference close to data.
  [0.5300] Embedding models in ONNX format can be loaded directly into Oracle Database.
  [0.7116] Oracle AI Database supports AI Vector Search for semantic retrieval.

QUERY: How does LangChain work with Oracle vectors?
----------------------------------------------------------------------------------------------------
  [0.2782]

## Step 10 — LangChain Integration with OracleEmbeddings

`langchain-oracledb` provides `OracleEmbeddings`, which wraps the in-database ONNX model. This means LangChain uses Oracle's `VECTOR_EMBEDDING()` under the hood — no external API calls.

In [15]:
from langchain_oracledb.embeddings import OracleEmbeddings

oracle_embedder = OracleEmbeddings(
    conn=conn,
    params={"provider": "database", "model": MODEL_NAME}
)

lc_embedding = oracle_embedder.embed_query("Oracle AI Database performs semantic search using vectors.")

print(f"Embedding dimension: {len(lc_embedding)}")
print(f"First 5 values:      {lc_embedding[:5]}")

Embedding dimension: 384
First 5 values:      [0.0141977277, -0.0539071336, -0.0459057614, -0.0626048148, -0.0612692609]


## Step 11 — Vector Store with OracleVS

`OracleVS` provides a LangChain-compatible vector store backed by Oracle AI Vector Search. Combined with `OracleEmbeddings`, the entire pipeline — embedding, storage, and retrieval — runs inside Oracle.

In [16]:
from langchain_core.documents import Document
from langchain_oracledb.vectorstores.oraclevs import OracleVS
from langchain_community.vectorstores.utils import DistanceStrategy

langchain_docs = [
    Document(page_content="Oracle AI Database supports vector storage and semantic search."),
    Document(page_content="An ONNX embedding model can be loaded directly into Oracle."),
    Document(page_content="LangChain can use OracleVS to query Oracle AI Vector Search."),
    Document(page_content="Using in-database embeddings can reduce architectural complexity."),
]

vector_store = OracleVS.from_documents(
    documents=langchain_docs,
    embedding=oracle_embedder,
    client=conn,
    table_name="LC_ONNX_DEMO",
    distance_strategy=DistanceStrategy.COSINE
)

print("OracleVS vector store created with in-database ONNX embeddings.")

OracleVS vector store created with in-database ONNX embeddings.


## Step 12 — Similarity Search Through LangChain

The query is embedded inside Oracle and compared against stored vectors — all through the LangChain abstraction.

In [17]:
results = vector_store.similarity_search(
    "How can Oracle Database help with semantic retrieval?",
    k=3
)

for i, doc in enumerate(results, start=1):
    print(f"{i}. {doc.page_content}")

1. Oracle AI Database supports vector storage and semantic search.
2. LangChain can use OracleVS to query Oracle AI Vector Search.
3. An ONNX embedding model can be loaded directly into Oracle.


## Step 13 — Inspect the Underlying Tables

Even through the LangChain abstraction, data lives in Oracle tables with `VECTOR` columns.

In [18]:
df_tables = run_sql(
    """
    SELECT table_name
    FROM user_tables
    WHERE table_name IN ('ONNX_DOCS', 'LC_ONNX_DEMO')
    ORDER BY table_name
    """,
    fetch=True
)
df_tables

,TABLE_NAME
0,LC_ONNX_DEMO
1,ONNX_DOCS


## Production Considerations

For production workloads:

- Create **vector indexes** (HNSW or IVF) for large-scale similarity search
- Use a larger, domain-specific document corpus
- Add **metadata filters** alongside semantic retrieval
- Consider quantized ONNX models for lower memory usage (`ONNXPipelineConfig(quantize_model=True)`)
- Monitor embedding quality by comparing retrieval relevance across different models

## Cleanup

These cells drop the demo tables and optionally remove the ONNX model from Oracle.

In [19]:
cleanup_statements = [
    """
    BEGIN
        EXECUTE IMMEDIATE 'DROP TABLE onnx_docs PURGE';
    EXCEPTION WHEN OTHERS THEN IF SQLCODE != -942 THEN RAISE; END IF;
    END;
    """,
    """
    BEGIN
        EXECUTE IMMEDIATE 'DROP TABLE LC_ONNX_DEMO PURGE';
    EXCEPTION WHEN OTHERS THEN IF SQLCODE != -942 THEN RAISE; END IF;
    END;
    """
]

for stmt in cleanup_statements:
    run_sql(stmt)

print("Demo tables dropped.")

# Uncomment to also remove the ONNX model:
# run_sql("BEGIN DBMS_VECTOR.DROP_ONNX_MODEL(model_name => :m, force => true); END;",
#         params={"m": MODEL_NAME})
# print(f"Model '{MODEL_NAME}' removed from Oracle.")

Demo tables dropped.


## Summary

This notebook demonstrated the **Oracle-first ONNX embedding workflow**:

1. **Model loading** — The augmented all-MiniLM-L12-v2 ONNX model was loaded into Oracle with `DBMS_VECTOR.LOAD_ONNX_MODEL`
2. **In-database embeddings** — `VECTOR_EMBEDDING()` generated 384-dimensional vectors directly in SQL, with no external API
3. **Semantic search** — `VECTOR_DISTANCE()` ranked documents by cosine similarity inside Oracle
4. **LangChain integration** — `OracleEmbeddings` and `OracleVS` provided a developer-friendly abstraction over the same in-database workflow

The key architectural advantage: **embedding generation, storage, and retrieval all happen inside Oracle Database**, reducing network round-trips and external dependencies.

## Learn More

- [Oracle AI Vector Search Documentation](https://docs.oracle.com/en/database/oracle/oracle-database/26/vecse/)
- [ONNX Pipeline Models for Text Embedding](https://docs.oracle.com/en/database/oracle/oracle-database/26/vecse/onnx-pipeline-models-text-embedding.html)
- [Pre-built ONNX Models Download](https://docs.oracle.com/pls/topic/lookup?ctx=en/database/oracle/oracle-database/26/vecse&id=oml_ai_models_object_storage)
- [LangChain Oracle Integration](https://python.langchain.com/docs/integrations/vectorstores/oracle/)
- [OML4Py ONNXPipeline for Custom Models](https://docs.oracle.com/en/database/oracle/machine-learning/oml4py/2/mlugp/convert-trained-models-onnx-format.html)